In [0]:
MY_ID = "arzoo"          # use your real ID
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"
S = f"workspace.capstone_{MY_ID}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {S}.raw")

DataFrame[]

In [0]:
import pyspark.sql.functions as F
A="array('Streetlight','Garbage','Water Supply','Drainage','Road Repair',"
A+="'Stray Animals','Property Tax','Encroachment')"
M="array(18.0,30.0,48.0,96.0,240.0,12.0,72.0,360.0)"
w=spark.range(1,41).selectExpr("concat('W',lpad(id,2,'0')) ward_id",
"concat('Z',cast(id%5 as int)) zone")
w.write.mode('overwrite').option('header',True).csv(f'{VOL}/raw/wards')
c=(spark.range(1,60001).selectExpr("cast((id-1) div 7500 as int) ix",
"cast(id%600 as int) r","concat('C',lpad(id,6,'0')) complaint_id",
"concat('W',lpad(cast((id-1)%40+1 as int),2,'0')) ward_id",
"timestampadd(minute,cast(rand(17)*217440 as int),timestamp'2025-01-01') raised_ts")
.selectExpr("*",f"element_at({A},ix+1) category",
f"element_at({M},ix+1)*exp(randn(2)*0.6) res_h")
.selectExpr("*","timestampadd(minute,cast(res_h*60 as int),raised_ts) c_ts")
.selectExpr("*","case when r%20 in (3,11,17) then 'OPEN'"
" when c_ts>timestamp'2025-06-29 23:59:59' then 'OPEN' else 'CLOSED' end st0")
.selectExpr("complaint_id","ward_id","category","r",
"case when r between 40 and 48 then 'CLOSED'"
" when r<=13 and r not in (3,11) then 'CLOSED' else st0 end status",
"cast(if(r between 20 and 26,timestamp'1970-01-05 00:00:00',raised_ts)"
" as string) raised_ts",
"case when r between 40 and 48 then cast(timestampadd(minute,"
"-60-cast(rand(3)*4260 as int),raised_ts) as string)"
" when r<=13 and r not in (3,11) then 'NA'"
" when st0='OPEN' then '' else cast(c_ts as string) end closed_ts"))
o=(c.filter('r between 60 and 64').withColumn('ward_id',F.lit('W99'))
.withColumn('complaint_id',F.concat(F.lit('C9'),F.expr("substr(complaint_id,3)"))))
out=c.unionByName(c.filter('r between 80 and 97')).unionByName(o).drop('r')
out.write.mode('overwrite').option('header',True).csv(f'{VOL}/raw/complaints')

In [0]:
print(spark.read.option("header", True).csv(f"{VOL}/raw/complaints").count())  # must be 62300
print(spark.read.option("header", True).csv(f"{VOL}/raw/wards").count())       # must be 40

62300
40
